# Pathfinding Algorithm Comparison

  Test and compare all pathfinding algorithms from `pathfinding_lite.py` on weighted maritime graphs.

  #### Algorithms Under Test

  | Algorithm | Heuristic | Pass Strategy |
  |-----------|-----------|---------------|
  | **Astar** | Haversine (great-circle) | Single-pass A* |
  | **AstarImproved** | Haversine + pilot-quantity (path straightness) | Single-pass A* |
  | **AstarMaritime** | Haversine scout + Dijkstra optimizer | Two-pass corridor (A* → corridor → Dijkstra) |
  | **AstarMaritimeSmooth** | Haversine scout + Dijkstra optimizer + string-pull smoothing | Three-pass corridor (A* → corridor → Dijkstra → string-pull) |

  #### Workflow

  1. **Configuration** — Select backend (GeoPackage / PostGIS), graph, vessel params, route endpoints
  2. **Load Weighted Graph** — Load pre-built directed weighted graph into NetworkX
  3. **Run All Algorithms** — Execute each with identical start/end points and timing
  4. **Comparison Dashboard** — Unified summary table with distance, time, edge stats
  5. **Visualization** — Interactive map with all routes overlaid plus S-57 layers (fairwy, tssplt, wrecks, etc.)
  6. **Detailed Analysis** — Per-algorithm edge weight breakdowns

  #### Outputs
  - Per-algorithm timing, distance, edge-factor counts
  - Side-by-side comparison table, bar charts, and normalized view
  - Interactive map with all routes overlaid plus S-57 context layers (TSS, wrecks, fairways)
  - Edge weight distribution histograms and factor breakdowns
  - Pairwise route difference analysis with Jaccard similarity



  #### Prerequisites
  - A weighted, directed graph already built (see `graph_weighted_directed_GeoPackage_v3.ipynb` or `graph_weighted_open_Postgis.ipynb`)
  - S-57 ENC data source (GeoPackage or PostGIS) for layer visualization

## 1. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# --- Backend Selection ---
# "geopackage" or "postgis"
BACKEND = "postgis"

# --- GeoPackage paths (used when BACKEND="geopackage") ---
graph_name_directed = "test_graph_directed_mem_v24o"
enc_data_file_name = "enc_west.gpkg"  # Relative to project /data

# --- PostGIS settings (used when BACKEND="postgis") ---
pg_graph_name = "test_graph_directed_postgis_v24w"
pg_schema = "enc_west"
pg_graph_schema = "graph"
pg_grid_schema = 'grid'

# --- Route Endpoints ---
# --- Pathfinding Ports ---
port_set = 1
reverse_route = False
custom_port_entry = "skip"

PORTS = {
      1: {
          "departure": {"name": "SF AIS Departure", "coords": {"lon": -122.09198, "lat": 38.04956}}, # Starting point name (from port database or custom)
          "arrival":   {"name": "SF Pilot",         "coords": {"lon": -122.780,   "lat": 37.006}},   # Ending point name
      },
      2: {
          "departure": {"name": "LA West TSS",      "coords": {"lon": -121.177,  "lat": 34.50}},
          "arrival":   {"name": "LA AIS Arrival",    "coords": {"lon": -118.26749,"lat": 33.76037}},
      },
      3: {
          "departure": {"name": "SF AIS Departure", "coords": {"lon": -122.09198, "lat": 38.04956}},
          "arrival":   {"name": "LA AIS Arrival",    "coords": {"lon": -118.26749,"lat": 33.76037}},
      },
  }

departure_port, arrival_port = PORTS[port_set]["departure"], PORTS[port_set]["arrival"]
if reverse_route:
    departure_port, arrival_port = arrival_port, departure_port


# --- AstarMaritime tuning ---
corridor_buffer_nm = 5.0
include_tss = True
tss_bbox_extend_factor = 0.5

# --- S-57 layers to overlay on the route map ---
viz_layers = {
    "tsslpt": {"color": "cyan",    "opacity": 0.4},
    "fairwy": {"color": "lime",    "opacity": 0.4},
    "wrecks": {"color": "red",     "opacity": 0.5},
    "obstrn": {"color": "orange",  "opacity": 0.5},
    "lndare": {"color": "tan",     "opacity": 0.35},
}

# --- Route display colors per algorithm ---
algo_colors = {
    "Astar":               "royalblue",      # darker than dodgerblue
    "AstarImproved":       "forestgreen",    # darker than limegreen
    "AstarMaritime":       "firebrick",      # darker than orangered
    "AstarMaritimeSmooth": "orange"
}

# --- Weight key for pathfinding ---
weight_key = 'adjusted_weight'

# --- Vessel parameters (reference for export metadata) ---
vessel_params = {
    'draft': 7.5, 'height': 30.0,
    'ukc_safety_margin': 2.0, 'ver_clearance_margin': 3.0,
    'vessel_type': 'cargo',
}

print(f"Backend: {BACKEND.upper()}")
print(f"Graph: {graph_name_directed if BACKEND == 'geopackage' else pg_graph_name}")
print(f"Weight key: {weight_key}")
print(f"Corridor buffer: {corridor_buffer_nm} NM")

### 1.1 Imports & Setup

In [ ]:
import os, sys, time, logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from collections import OrderedDict

import geopandas as gpd
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv
from shapely.geometry import Point, LineString
from shapely.ops import unary_union

# --- Fix PROJ_LIB Path (Common Conda/Jupyter Issue) ---
# Ensure GDAL/PROJ can find the coordinate database
conda_prefix = sys.prefix
possible_proj_lib = os.path.join(conda_prefix, 'share', 'proj')
if os.path.exists(possible_proj_lib):
    os.environ['PROJ_LIB'] = possible_proj_lib

project_root = Path.cwd().parent.parent
load_dotenv(project_root / ".env")

# --- Project Imports ---
from nautical_graph_toolkit.core.graph import H3Graph
from nautical_graph_toolkit.core.s57_data import ENCDataFactory
from nautical_graph_toolkit.core.pathfinding_lite import (
    Astar, AstarImproved, AstarMaritime, AstarMaritimeSmooth, Route,
)
from nautical_graph_toolkit.utils.plot_utils import PlotlyChart
from nautical_graph_toolkit.utils.port_utils import PortData

# --- Setup ---
output_dir = Path.cwd() / 'output'
output_dir.mkdir(exist_ok=True)
ply = PlotlyChart()
port_manager = PortData()
mapbox_token = os.getenv('MAPBOX_TOKEN')

# Create or update the custom port locations
for ports in (departure_port, arrival_port):
      if port_manager.get_port_by_name(ports['name']) is None:
          port_manager.create_custom_port(
              port_name=ports['name'],
              lon=ports['coords']['lon'],
              lat=ports['coords']['lat'],
              if_exists=custom_port_entry,
          )
          print(f"New Custom Port {ports['name']} with {ports['coords']} registered")
      else:
          print(f"Port {ports['name']} with {ports['coords']} already registered")

logging.basicConfig(level=logging.WARNING)
logging.getLogger('nautical_graph_toolkit').setLevel(logging.INFO)

print("Setup complete.")


## 2. Load Weighted Graph
Load the fully-weighted directed graph into a NetworkX `nx.Graph`. This is the same graph produced by the weighting pipeline (static + directional + dynamic tiers combined into `adjusted_weight`).

In [ ]:
if BACKEND == "geopackage":
    graph_name = graph_name_directed
else:
    graph_name = pg_graph_name

print(f"Loading weighted directed graph '{graph_name}'...")
t0 = time.perf_counter()

def _load_graph():
    """Load the weighted directed graph from the configured backend."""
    if BACKEND == "geopackage":
        gpkg_path = output_dir / f"{graph_name_directed}.gpkg"
        if not gpkg_path.exists():
            raise FileNotFoundError(f"Graph not found: {gpkg_path}")
        factory = ENCDataFactory(source=project_root / "data" / enc_data_file_name)
        graph_mgr = H3Graph(data_factory=factory)
        G = graph_mgr.load_graph_from_gpkg(str(gpkg_path), directed=True)
        return G, factory

    elif BACKEND == "postgis":
        db_params = {
            'dbname': os.getenv('DB_NAME'), 'user': os.getenv('DB_USER'),
            'password': os.getenv('DB_PASSWORD'), 'host': os.getenv('DB_HOST'),
            'port': os.getenv('DB_PORT'),
        }
        factory = ENCDataFactory(source=db_params, schema=pg_schema)
        graph_mgr = H3Graph(data_factory=factory, graph_schema_name=pg_graph_schema)
        G = graph_mgr.load_graph_from_postgis(table_prefix=pg_graph_name, directed=True)
        return G, factory
    else:
        raise ValueError(f"Unknown BACKEND: {BACKEND}")

G, factory = _load_graph()

load_time = time.perf_counter() - t0

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Backend: {BACKEND}")

# --- Weight statistics ---
weights = [d.get(weight_key, d.get('weight', 0)) for _, _, d in G.edges(data=True)]
if weights:
    print(f"\nWeight statistics ({weight_key}):")
    print(f"  Min:    {min(weights):.6f}")
    print(f"  Max:    {max(weights):.6f}")
    print(f"  Mean:   {np.mean(weights):.6f}")
    print(f"  Median: {np.median(weights):.6f}")


## 3. Resolve Departure / Arrival Points

In [ ]:
dep_port = port_manager.get_port_by_name(departure_port['name'])
arr_port = port_manager.get_port_by_name(arrival_port['name'])

departure_point = dep_port.geometry
arrival_point   = arr_port.geometry

print(f"Departure: {departure_port['name']} ({departure_point.x:.4f}, {departure_point.y:.4f})")
print(f"Arrival:   {arrival_port['name']} ({arrival_point.x:.4f}, {arrival_point.y:.4f})")


## 4. Run All Pathfinding Algorithms
Each algorithm uses `weight_key='adjusted_weight'` and runs on the **same graph** with the **same endpoints**. Times are measured with `time.perf_counter()`.

### Unified Metrics for Any Algorithm
`AstarMaritime` already exposes `get_maritime_metrics()`. The helper below computes the same metrics for `Astar` and `AstarImproved` so all three are directly comparable.

In [ ]:
def compute_unified_metrics(
    graph,
    route_geom: LineString,
    path_nodes: List[Tuple[float, float]],
    elapsed_s: float,
    weight_key: str = weight_key,
    algo_name: str = '',
) -> Dict[str, Any]:
    total_dist = Route._calculate_route_distance(route_geom)

    accumulated_weight = 0.0
    blocking_count = 0
    penalty_count = 0
    bonus_count = 0
    dir_opposite_count = 0
    shortcut_count = 0

    for i in range(len(path_nodes) - 1):
        u, v = path_nodes[i], path_nodes[i + 1]
        if graph.has_edge(u, v):
            ed = graph[u][v]
            accumulated_weight += ed.get(weight_key, ed.get('weight', 0.0))
            if ed.get('blocking_factor', 0) >= 1000:
                blocking_count += 1
            if ed.get('penalty_factor', 1.0) > 1.0:
                penalty_count += 1
            if ed.get('bonus_factor', 1.0) < 1.0:
                bonus_count += 1
            if ed.get('wt_dir', 1.0) >= 50.0:
                dir_opposite_count += 1
        else:
            shortcut_count += 1

    return OrderedDict({
        'algorithm':          algo_name,
        'total_distance_nm':  round(total_dist, 4),
        'num_nodes':          len(path_nodes),
        'num_edges':          len(path_nodes) - 1,
        'shortcut_count':     shortcut_count,
        'accumulated_weight': round(accumulated_weight, 4),
        'blocking_count':     blocking_count,
        'penalty_count':      penalty_count,
        'bonus_count':        bonus_count,
        'dir_opposite_count': dir_opposite_count,
        'computation_time_s': round(elapsed_s, 4),
        'route_geom':         route_geom,
        'path_nodes':         path_nodes,
    })


def run_maritime_route(
    route_finder, dep_point, arr_point,
    astar_impl, algo_name: str,
    weight_key: str = weight_key,
    **kwargs,
) -> Optional[Dict[str, Any]]:
    """Run a maritime algorithm and return unified metrics with maritime extras."""
    t0 = time.perf_counter()
    result = route_finder.detailed_route(
        dep_point, arr_point,
        astar_impl=astar_impl,
        weight_key=weight_key,
        min_cost_factor=1.0,
        **kwargs,
    )
    elapsed = time.perf_counter() - t0

    if not result:
        return None

    geom = result['route_geometry']
    raw  = result.get('maritime_metrics', {})
    pf   = route_finder._last_pathfinder

    nodes = pf._pass2_path_nodes or list(geom.coords)[1:-1]
    metrics = compute_unified_metrics(G, geom, nodes, elapsed, algo_name=algo_name)

    # Maritime common extras
    metrics['pass_used']          = raw.get('pass_used', '')
    metrics['pass1_distance_nm'] = raw.get('pass1_distance_nm', 0)
    metrics['pass2_distance_nm'] = raw.get('pass2_distance_nm', 0)
    metrics['corridor_stats']    = raw.get('corridor_stats', {})
    metrics['computation_detail']= raw.get('computation_time_s', {})

    # String-pull extras (AstarMaritimeSmooth only)
    if 'sp_original_nodes' in raw:
        metrics['sp_original_nodes'] = raw['sp_original_nodes']
        metrics['sp_smoothed_nodes'] = raw['sp_smoothed_nodes']
        metrics['sp_reduction_pct']  = raw.get('sp_reduction_pct', 0.0)

    return metrics


def print_maritime_summary(metrics: Dict[str, Any]):
    """Print unified summary for any maritime-class result."""
    print(f"  Distance: {metrics['total_distance_nm']:.2f} NM | "
          f"Edges: {metrics['num_edges']} | Time: {metrics['computation_time_s']:.3f}s")

    if metrics.get('pass1_distance_nm', 0) > 0:
        print(f"  Pass used: {metrics['pass_used']} | "
              f"Pass1: {metrics['pass1_distance_nm']:.2f} NM → "
              f"Pass2: {metrics['pass2_distance_nm']:.2f} NM")

    cs = metrics.get('corridor_stats')
    if cs:
        print(f"  Corridor: {cs.get('subgraph_nodes', 0):,} nodes / "
              f"{cs.get('subgraph_edges', 0):,} edges "
              f"(from {cs.get('full_graph_nodes', 0):,} nodes)")

    sp_orig = metrics.get('sp_original_nodes', 0)
    if sp_orig > 0:
        print(f"  String-pull: {sp_orig} → {metrics['sp_smoothed_nodes']} nodes "
              f"({metrics['sp_reduction_pct']:.1f}% reduction)")


### 4.1 Run Astar (Basic)

In [ ]:
print("=" * 60)
print("Running ALGORITHM 1: A* (Basic haversine)")
print("=" * 60)

route_finder = Route(graph=G, data_manager=factory.manager)

t0 = time.perf_counter()
result_basic = route_finder.detailed_route(
    departure_point, arrival_point,
    astar_impl=Astar,
    weight_key=weight_key,
    min_cost_factor=1.0,
)
t_basic = time.perf_counter() - t0

if result_basic:
    geom_basic = result_basic['route_geometry']
    nodes_basic = list(geom_basic.coords)[1:-1]  # strip start/end user points
    metrics_basic = compute_unified_metrics(G, geom_basic, nodes_basic, t_basic, algo_name='Astar')
    print(f"  Distance: {metrics_basic['total_distance_nm']:.2f} NM | "
          f"Edges: {metrics_basic['num_edges']} | Time: {t_basic:.3f}s")
else:
    metrics_basic = None
    print("  No route found.")

### 4.2 Run AstarImproved (pilot-quantity heuristic)

In [ ]:
print("=" * 60)
print("Running ALGORITHM 2: A* Improved (pilot-quantity) ")
print("=" * 60)

t0 = time.perf_counter()
result_improved = route_finder.detailed_route(
    departure_point, arrival_point,
    astar_impl=AstarImproved,
    weight_key=weight_key,
    min_cost_factor=1.0,
)
t_improved = time.perf_counter() - t0

if result_improved:
    geom_improved = result_improved['route_geometry']
    nodes_improved = list(geom_improved.coords)[1:-1]
    metrics_improved = compute_unified_metrics(G, geom_improved, nodes_improved, t_improved, algo_name='AstarImproved')
    print(f"  Distance: {metrics_improved['total_distance_nm']:.2f} NM | "
          f"Edges: {metrics_improved['num_edges']} | Time: {t_improved:.3f}s")
else:
    metrics_improved = None
    print("  No route found.")

### 4.3 Run AstarMaritime (Two-Pass Corridor)

In [ ]:
print("=" * 60)
print("Running ALGORITHM 3: A* Maritime (two-pass corridor)")
print("=" * 60)

metrics_maritime = run_maritime_route(
    route_finder, departure_point, arrival_point,
    astar_impl=AstarMaritime,
    algo_name="AstarMaritime",
    corridor_buffer_nm=corridor_buffer_nm,
    include_tss=include_tss,
    tss_bbox_extend_factor=tss_bbox_extend_factor,
)

if metrics_maritime:
    print_maritime_summary(metrics_maritime)
else:
    print("  No route found.")


### 4.4 Run AstarMaritimeSmooth (Three-Pass String Pulling)

In [ ]:
print("=" * 60)
print("Running ALGORITHM 4: A* Maritime (three-pass string pulling)")
print("=" * 60)

metrics_smooth = run_maritime_route(
    route_finder, departure_point, arrival_point,
    astar_impl=AstarMaritimeSmooth,
    algo_name="AstarMaritimeSmooth",
    corridor_buffer_nm=corridor_buffer_nm,
    include_tss=include_tss,
    tss_bbox_extend_factor=tss_bbox_extend_factor,
    apply_smoothing=True,
    sp_buffer_nm=0.15,
)

if metrics_smooth:
    print_maritime_summary(metrics_smooth)
else:
    print("  No route found.")


## 5. Comparison Dashboard

In [ ]:
# ── Collect valid results ──
all_metrics = [m for m in [metrics_basic, metrics_improved, metrics_maritime, metrics_smooth] if m is not None]

# ── Summary table ──
compare_cols = [
    'algorithm', 'total_distance_nm', 'num_edges', 'accumulated_weight',
    'blocking_count', 'penalty_count', 'bonus_count', 'dir_opposite_count',
    'computation_time_s',
]
df_compare = pd.DataFrame([{k: m[k] for k in compare_cols} for m in all_metrics])
df_compare = df_compare.set_index('algorithm')
df_compare = df_compare.sort_values('computation_time_s')

print("\n" + "=" * 90)
print("ALGORITHM COMPARISON")
print("=" * 90)
display(df_compare.style.format({
    'total_distance_nm': '{:.2f}',
    'accumulated_weight': '{:.2f}',
    'computation_time_s': '{:.3f}',
}))

# ── Best-in-class ──
fastest = df_compare['computation_time_s'].min()
shortest = df_compare['total_distance_nm'].min()
lightest = df_compare['accumulated_weight'].min()
print(f"\nFastest: {fastest:.3f}s | Shortest: {shortest:.2f} NM | Lowest weight: {lightest:.2f}")

### 5.1 Comparison Bar Charts

In [ ]:
if len(all_metrics) >= 2:
    names = [m['algorithm'] for m in all_metrics]

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=('Distance (NM)', 'Computation Time (s)', 'Accumulated Weight',
                        'Blocked Edges', 'Penalty Edges', 'Bonus Edges'),
        vertical_spacing=0.15
    )

    metrics_keys = [
        ('total_distance_nm', 1, 1),
        ('computation_time_s', 1, 2),
        ('accumulated_weight', 1, 3),
        ('blocking_count', 2, 1),
        ('penalty_count', 2, 2),
        ('bonus_count', 2, 3),
    ]

    for key, row, col in metrics_keys:
        vals = [m[key] for m in all_metrics]
        colors = [algo_colors.get(n, 'gray') for n in names]
        fig.add_trace(
            go.Bar(x=names, y=vals, marker_color=colors, showlegend=False),
            row=row, col=col
        )

    fig.update_layout(
        title_text="Algorithm Comparison Dashboard",
        height=700,
        template='plotly_dark'

    )
    fig.show()


### 5.2 Normalized comparison

In [ ]:
# ── Normalized comparison (each metric as % of worst) ──
if len(all_metrics) >= 2:
    norm_cols = ['total_distance_nm', 'num_edges', 'accumulated_weight', 'computation_time_s']
    df_norm = df_compare[norm_cols].copy()
    for c in norm_cols:
        df_norm[c] = df_norm[c] / df_norm[c].max() * 100

    display(df_norm.style.format('{:.1f}').set_caption('Normalized (% of worst)'))

## 6. Visualization — Routes with S-57 Layers

### 6.1 Route Visualization
All four routes plotted on a single interactive map. Toggle legend entries to compare.

In [ ]:
fig_map = ply.create_base_map(
    title="Pathfinding Algorithm Comparison",
    mapbox_token=mapbox_token,
)

# ── Plot each route ──
for m in all_metrics:
    name = m['algorithm']
    color = algo_colors.get(name, 'gray')
    width = 5 if name == 'AstarMaritime' else 4
    ply.add_route_trace(
        fig_map,
        line=m['route_geom'],
        name=f"{name} ({m['total_distance_nm']:.1f} NM)",
        color=color,
        width=width,
    )

# ── Departure / arrival markers ──
for pt, label, marker_color in [
    (departure_point, departure_port['name'], 'blue'),
    (arrival_point, arrival_port['name'], 'red'),
]:
    fig_map.add_trace(go.Scattermapbox(
        lon=[pt.x], lat=[pt.y],
        mode='markers+text',
        marker=dict(size=14, color=marker_color),
        text=[label], textposition='top right',
        name=label, showlegend=True,
    ))

fig_map.show()


### 6.2 Interactive Map with All Routes & Layers
Overlay TSS lanes, wrecks, fairways, obstructions, and land areas under the routes to understand routing decisions.

In [ ]:
 # ── Get ENC list from route bounds ──
all_geoms = [m['route_geom'] for m in all_metrics]
combined_bounds = unary_union(all_geoms).convex_hull
try:
    enc_list = factory.get_encs_by_boundary(combined_bounds)
    print(f"Using {len(enc_list)} ENCs for layer visualization")
except Exception as e:
    print(f"ENC boundary detection failed ({e}) — skipping S-57 layers")
    enc_list = []

# ── Build figure ──
fig_layers = ply.create_base_map(
    title="Routes with S-57 Layers",
    mapbox_token=mapbox_token,
)

# ── Add S-57 layers (underneath routes) ──
if not enc_list:
    print("No ENCs resolved — layer overlay skipped")
else:
    for layer_name, layer_cfg in viz_layers.items():
        try:
            layer_gdf = factory.get_layer(layer_name=layer_name, filter_by_enc=enc_list)
            if layer_gdf is None or layer_gdf.empty:
                print(f"  {layer_name}: no features, skipped")
                continue
            ply.add_layer_trace(
                fig_layers,
                layer_df=layer_gdf,
                name=layer_name.upper(),
                color=layer_cfg['color'],
                fill_opacity=layer_cfg['opacity'],
            )
            print(f"  {layer_name}: {len(layer_gdf)} features added")
        except Exception as e:
            print(f"  {layer_name}: skipped ({e})")

# ── Re-add routes on top ──
for m in all_metrics:
    name = m['algorithm']
    color = algo_colors.get(name, 'gray')
    ply.add_route_trace(
        fig_layers,
        line=m['route_geom'],
        name=f"{name} ({m['total_distance_nm']:.1f} NM)",
        color=color,
        width=3,
    )

# ── Port markers ──
for pt, label, marker_color in [
    (departure_point, departure_port['name'], 'blue'),
    (arrival_point, arrival_port['name'], 'red'),
]:
    fig_layers.add_trace(go.Scattermapbox(
        lon=[pt.x], lat=[pt.y],
        mode='markers+text',
        marker=dict(size=14, color=marker_color),
        text=[label], textposition='top right',
        name=label, showlegend=True,
    ))

fig_layers.show()

## 7. Detailed Analysis

### 7.1 Detailed Edge Analysis
Per-algorithm weight distributions and factor breakdowns along each route.

In [ ]:
if all_metrics:
    fig_hist = make_subplots(
        rows=1, cols=len(all_metrics),
        subplot_titles=[m['algorithm'] for m in all_metrics],
        shared_yaxes=True
    )

    for idx, m in enumerate(all_metrics):
        path = m['path_nodes']
        edge_weights = []
        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            if G.has_edge(u, v):
                edge_weights.append(G[u][v].get(weight_key, G[u][v].get('weight', 0)))

        if edge_weights:
            color = algo_colors.get(m['algorithm'], 'gray')
            fig_hist.add_trace(
                go.Histogram(
                    x=edge_weights,
                    nbinsx=50,
                    name=m['algorithm'],
                    marker_color=color,
                    showlegend=True,
                ),
                row=1, col=idx + 1
            )

    fig_hist.update_layout(
        title_text="Edge Weight Distribution Along Routes",
        height=450,
        template='plotly_dark',
        xaxis_title="Weight",
        yaxis_title="Edge Count"
    )
    fig_hist.show()

### 7.2 Factor Breakdown Table

In [ ]:
if all_metrics:
    print("=" * 70)
    print("FACTOR BREAKDOWN PER ALGORITHM")
    print("=" * 70)

    for m in all_metrics:
        path = m['path_nodes']
        total_edges = m['num_edges']
        shortcut_edges = m.get('shortcut_count', 0)

        blocking_factors, penalty_factors, bonus_factors, dir_weights = [], [], [], []

        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            if G.has_edge(u, v):
                ed = G[u][v]
                bf = ed.get('blocking_factor', 1.0)
                pf = ed.get('penalty_factor', 1.0)
                bnf = ed.get('bonus_factor', 1.0)
                dw = ed.get('wt_dir', 1.0)
                if bf > 1.0: blocking_factors.append(bf)
                if pf > 1.0: penalty_factors.append(pf)
                if bnf < 1.0: bonus_factors.append(bnf)
                if dw != 1.0: dir_weights.append(dw)

        print(f"\n--- {m['algorithm']} ({total_edges} edges) ---")
        print(f"  Distance:              {m['total_distance_nm']:.2f} NM")
        print(f"  Accumulated Weight:     {m['accumulated_weight']:.4f}")
        print(f"  Computation Time:       {m['computation_time_s']:.2f}s")

        if shortcut_edges:
            print(f"  Graph edges:            {total_edges - shortcut_edges} | Shortcut edges: {shortcut_edges}")

        if blocking_factors:
            print(f"  Blocking edges:         {len(blocking_factors)} "
                  f"(mean factor: {np.mean(blocking_factors):.1f}, "
                  f"max: {max(blocking_factors):.0f})")
        else:
            print(f"  Blocking edges:         0")

        if penalty_factors:
            print(f"  Penalty edges:          {len(penalty_factors)} "
                  f"(mean factor: {np.mean(penalty_factors):.2f})")
        else:
            print(f"  Penalty edges:          0")

        if bonus_factors:
            print(f"  Bonus edges:            {len(bonus_factors)} "
                  f"(mean factor: {np.mean(bonus_factors):.3f})")
        else:
            print(f"  Bonus edges:            0")

        if dir_weights:
            print(f"  Directional edges:      {len(dir_weights)} "
                  f"(mean wt_dir: {np.mean(dir_weights):.2f})")

        # Maritime details (AstarMaritime / AstarMaritimeSmooth)
        if 'pass_used' in m:
            print(f"  Pass used:              {m['pass_used']}")
            if m.get('pass1_distance_nm', 0) > 0:
                delta = m['pass1_distance_nm'] - m.get('pass2_distance_nm', m['total_distance_nm'])
                print(f"  Pass 1 distance:        {m['pass1_distance_nm']:.2f} NM")
                print(f"  Pass 2 distance:        {m.get('pass2_distance_nm', 'N/A')} NM")
                print(f"  Pass 1→2 improvement:   {delta:+.2f} NM")
            if m.get('corridor_stats'):
                cs = m['corridor_stats']
                reduction = 1 - cs['subgraph_edges'] / cs['full_graph_edges']
                print(f"  Corridor reduction:     {reduction:.1%} "
                      f"({cs['subgraph_edges']:,}/{cs['full_graph_edges']:,} edges)")

        # String-pull details (AstarMaritimeSmooth only)
        sp_orig = m.get('sp_original_nodes', 0)
        if sp_orig > 0:
            print(f"  String-pull reduction:  {sp_orig} → {m['sp_smoothed_nodes']} nodes "
                  f"({m['sp_reduction_pct']:.1f}% reduction)")


### 7.3 Route Difference Analysis
Compare routes pairwise with Jaccard similarity and distance deltas to quantify how much the algorithms actually diverge.

In [ ]:
if len(all_metrics) >= 2:
    print("=" * 70)
    print("ROUTE DIFFERENCE ANALYSIS")
    print("=" * 70)

    for i in range(len(all_metrics)):
        for j in range(i + 1, len(all_metrics)):
            m1, m2 = all_metrics[i], all_metrics[j]
            dist_diff = m2['total_distance_nm'] - m1['total_distance_nm']
            time_diff = m2['computation_time_s'] - m1['computation_time_s']
            weight_diff = m2['accumulated_weight'] - m1['accumulated_weight']

            # Jaccard similarity of edges
            edges1 = set()
            for k in range(len(m1['path_nodes']) - 1):
                edges1.add((m1['path_nodes'][k], m1['path_nodes'][k + 1]))
            edges2 = set()
            for k in range(len(m2['path_nodes']) - 1):
                edges2.add((m2['path_nodes'][k], m2['path_nodes'][k + 1]))

            shared = edges1 & edges2
            union = edges1 | edges2
            jaccard = len(shared) / len(union) if union else 0

            pct_str = (f"{dist_diff / m1['total_distance_nm'] * 100:+.1f}%"
                       if m1['total_distance_nm'] else "N/A")
            print(f"\n{m1['algorithm']}  vs  {m2['algorithm']}:")
            print(f"  Distance:   {m1['total_distance_nm']:.2f} vs {m2['total_distance_nm']:.2f} NM  "
                  f"(delta: {dist_diff:+.2f} NM, {pct_str})")
            print(f"  Time:       {m1['computation_time_s']:.2f} vs {m2['computation_time_s']:.2f}s  "
                  f"(delta: {time_diff:+.2f}s)")
            print(f"  Weight:     {m1['accumulated_weight']:.4f} vs {m2['accumulated_weight']:.4f}  "
                  f"(delta: {weight_diff:+.4f})")
            print(f"  Edge Jaccard similarity: {jaccard:.1%} ")

## 8. Export Comparison to CSV

In [ ]:
# ── Export comparison table ──
csv_path = output_dir / "pathfinding_comparison.csv"
df_compare.to_csv(csv_path)
print(f"Comparison table → {csv_path}")

# ── Export each route to GeoJSON ──
for m in all_metrics:
    name = m['algorithm']

    if reverse_route:
        geojson_path = output_dir / f"route_{name.lower()}_rev.geojson"
    else:
        geojson_path = output_dir / f"route_{name.lower()}.geojson"
    route_gdf = gpd.GeoDataFrame(
        [{
            'algorithm': name,
            'distance_nm': m['total_distance_nm'],
            'time_s': m['computation_time_s'],
            'edges': m['num_edges'],
            'accumulated_weight': m['accumulated_weight'],
            'vessel_draft': vessel_params['draft'],
            'vessel_type': vessel_params['vessel_type'],
        }],
        geometry=[m['route_geom']],
        crs='EPSG:4326',
    )
    route_gdf.to_file(geojson_path, driver='GeoJSON')
    print(f"  {name} route → {geojson_path.name}")
